# 04 — P3 Segmentation 결과 분석 (smp U-Net + ResNet34)

이 노트북은 **P3 segmentation_models_pytorch(smp) U-Net + ResNet34 실험** 결과를 분석합니다.
3-class (bg / normal / canker) 분할을 3 epoch 학습하여 mIoU = **0.940**을 달성했습니다.
학습 곡선, 픽셀 단위 Confusion Matrix, 정성 샘플, 실패 케이스, class별 성능 비교를 살펴봅니다.

## What you'll learn
- `train.log` 파싱으로 mIoU / pixelAccuracy 학습 곡선 그리기
- `metrics.json`의 confusion_matrix로 per-class IoU, Dice 계산
- seaborn heatmap으로 픽셀 단위 confusion matrix 시각화
- val 전체 추론 → per-image IoU → 실패 케이스 탐지
- segmentation 난이도 분석: polygon이 병변이 아닌 과일 외곽선인 의미


## Setup

In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path
PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import re, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import torch
import cv2

# 한글 폰트 설정 (macOS)
_korean_candidates = [
    "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
    "/Library/Fonts/NanumGothic.ttf",
]
for _fc in _korean_candidates:
    if Path(_fc).exists():
        fm.fontManager.addfont(_fc)
        matplotlib.rcParams["font.family"] = fm.FontProperties(fname=_fc).get_name()
        break
matplotlib.rcParams["axes.unicode_minus"] = False

# ── 최신 segmentation run 찾기 ───────────────────────────────────────────────
SEG_ROOT = PROJECT_ROOT / "outputs" / "segmentation" / "run"
run_dirs = sorted(SEG_ROOT.iterdir()) if SEG_ROOT.exists() else []

if not run_dirs:
    raise FileNotFoundError(f"No segmentation runs found under {SEG_ROOT}")

LATEST_RUN = run_dirs[-1]
print(f"Latest segmentation run: {LATEST_RUN.name}")
print(f"  train.log  : {(LATEST_RUN / 'train.log').exists()}")
print(f"  metrics.json: {(LATEST_RUN / 'metrics.json').exists()}")
print(f"  best.pt    : {(LATEST_RUN / 'ckpt' / 'best.pt').exists()}")
print(f"  qualitative: {list((LATEST_RUN / 'qualitative').glob('*.png')) if (LATEST_RUN / 'qualitative').exists() else '없음'}")


## 1. 학습 곡선

`train.log`에서 정규표현식으로 epoch별 train_loss / val_loss / miou / pixAcc를 파싱하고
3개 패널(losses, mIoU, pixelAccuracy)로 시각화합니다.


In [ ]:
log_path = LATEST_RUN / "train.log"

# 로그 패턴 예시:
#   epoch 1/3 train_loss=0.2900 val_loss=0.2056 miou=0.5425 pixAcc=0.9412 ...
epoch_pattern = re.compile(
    r"epoch\s+(\d+)/\d+\s+"
    r"train_loss=([\d.]+)\s+"
    r"val_loss=([\d.]+)\s+"
    r"miou=([\d.]+)\s+"
    r"pixAcc=([\d.]+)"
)

epochs, train_losses, val_losses, mious, pixaccs = [], [], [], [], []
with open(log_path) as f:
    for line in f:
        m = epoch_pattern.search(line)
        if m:
            epochs.append(int(m.group(1)))
            train_losses.append(float(m.group(2)))
            val_losses.append(float(m.group(3)))
            mious.append(float(m.group(4)))
            pixaccs.append(float(m.group(5)))

df_log = pd.DataFrame({
    "epoch":      epochs,
    "train_loss": train_losses,
    "val_loss":   val_losses,
    "miou":       mious,
    "pixAcc":     pixaccs,
})
print(df_log.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Loss ---
ax = axes[0]
ax.plot(df_log["epoch"], df_log["train_loss"], "o-", label="train_loss", color="steelblue")
ax.plot(df_log["epoch"], df_log["val_loss"],   "s--", label="val_loss",  color="darkorange")
ax.set_title("Train / Val Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)
ax.set_xticks(df_log["epoch"])

# --- mIoU ---
ax = axes[1]
ax.plot(df_log["epoch"], df_log["miou"], "^-", color="seagreen", linewidth=2)
ax.set_title("Val mIoU")
ax.set_xlabel("Epoch"); ax.set_ylabel("mIoU")
ax.set_ylim(0.45, 1.0); ax.grid(alpha=0.3)
ax.set_xticks(df_log["epoch"])

# --- pixelAccuracy ---
ax = axes[2]
ax.plot(df_log["epoch"], df_log["pixAcc"], "D-", color="tomato", linewidth=2)
ax.set_title("Val Pixel Accuracy")
ax.set_xlabel("Epoch"); ax.set_ylabel("Pixel Accuracy")
ax.set_ylim(0.9, 1.0); ax.grid(alpha=0.3)
ax.set_xticks(df_log["epoch"])

plt.suptitle("U-Net (ResNet34) 학습 곡선", fontsize=14)
plt.tight_layout()
plt.show()


## 2. 최종 Metrics (Best Checkpoint)

`metrics.json`에 저장된 최종 평가 지표를 테이블로 정리합니다.
3개 클래스: **bg (0), normal (1), canker (2)**.


In [ ]:
metrics_path = LATEST_RUN / "metrics.json"
with open(metrics_path) as f:
    metrics = json.load(f)

CLASS_NAMES = ["bg", "normal", "canker"]

# 요약 테이블
rows = []
rows.append({"Metric": "mIoU (all classes)",  "Value": f"{metrics['miou']:.4f}"})
rows.append({"Metric": "Pixel Accuracy",       "Value": f"{metrics['pixel_accuracy']:.4f}"})
for i, cls in enumerate(CLASS_NAMES):
    rows.append({"Metric": f"IoU  — {cls}", "Value": f"{metrics['iou_per_class'][i]:.4f}"})
for i, cls in enumerate(CLASS_NAMES):
    rows.append({"Metric": f"Dice — {cls}", "Value": f"{metrics['dice_per_class'][i]:.4f}"})

df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))


## 3. 픽셀 단위 Confusion Matrix

`metrics.json`의 `confusion_matrix` 필드는 (3×3) 행렬로 **행 = 실제 클래스, 열 = 예측 클래스**입니다.
정규화(row-normalize)된 버전도 함께 시각화합니다.


In [ ]:
cm = np.array(metrics["confusion_matrix"])  # shape (3, 3)
print("Confusion Matrix (pixel counts):")
df_cm = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
print(df_cm.to_string())

# row-normalize
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, fmt, title in [
    (axes[0], cm,      "d",    "Confusion Matrix (pixel 수)"),
    (axes[1], cm_norm, ".3f",  "Confusion Matrix (행 정규화)"),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=[f"pred:{c}" for c in CLASS_NAMES],
        yticklabels=[f"true:{c}" for c in CLASS_NAMES],
        ax=ax, linewidths=0.5, cbar=True
    )
    ax.set_title(title, fontsize=12)
    ax.set_ylabel("실제 클래스")
    ax.set_xlabel("예측 클래스")

plt.tight_layout()
plt.show()


## 4. 정성 샘플 (original | GT | pred)

학습 중 저장된 `qualitative/sample_XXX.png` 파일을 불러와 표시합니다.
각 PNG는 ultralytics 스타일과 유사하게 3-pane(원본 | GT mask | pred mask) composite 이미지입니다.


In [ ]:
qual_dir = LATEST_RUN / "qualitative"
qual_files = sorted(qual_dir.glob("sample_*.png")) if qual_dir.exists() else []

if not qual_files:
    print("[INFO] qualitative 샘플이 없습니다.")
    print("  (eval.save_qualitative_every_n_epochs가 학습 epoch 수보다 크면 저장 안 됨)")
else:
    n = len(qual_files)
    fig, axes = plt.subplots(n, 1, figsize=(16, 5 * n))
    if n == 1:
        axes = [axes]
    for ax, qpath in zip(axes, qual_files):
        img = np.array(plt.imread(str(qpath)))
        ax.imshow(img)
        ax.set_title(qpath.name, fontsize=11)
        ax.axis("off")
    plt.suptitle("정성 샘플 (학습 중 저장)", fontsize=14)
    plt.tight_layout()
    plt.show()


## 5. 실패 케이스 분석

best.pt를 로드하여 val 전체(88장)를 추론하고, per-image IoU(normal + canker 평균)를 계산합니다.
하위 3장을 worst case로 시각화합니다 (original | GT mask | pred mask).


In [ ]:
import yaml
from torch.utils.data import DataLoader
from segmentation.model import build_model
from segmentation.transforms import build_transforms
from common.dataset import SegmentationDataset

# config 로드
cfg_path = LATEST_RUN / "config.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else "cpu"
)
print("Device:", device)

# 모델 로드
seg_model = build_model(
    num_classes=cfg["model"]["num_classes"],
    encoder_name=cfg["model"]["encoder_name"],
    encoder_weights=None,   # 추론 시에는 가중치 로드 불필요
)
ckpt_path = LATEST_RUN / "ckpt" / "best.pt"
state = torch.load(ckpt_path, map_location=device, weights_only=True)
seg_model.load_state_dict(state)
seg_model = seg_model.to(device).eval()
print(f"모델 로드 완료: {ckpt_path}")

# val dataloader
img_size = cfg["data"]["image_size"]
val_tf = build_transforms(img_size, train=False)
val_ds = SegmentationDataset(PROJECT_ROOT / "database", split="val", transform=val_tf)
val_loader = DataLoader(
    val_ds, batch_size=4, shuffle=False, num_workers=0,
    collate_fn=lambda batch: {
        "image":      torch.stack([b["image"] for b in batch]),
        "mask":       torch.stack([b["mask"].long() for b in batch]),
        "image_path": [b["image_path"] for b in batch],
    }
)
print(f"Val 샘플 수: {len(val_ds)}")


In [ ]:
# per-image IoU 계산 (non-bg classes: 1=normal, 2=canker)
results_per_image = []  # [(mean_iou, img_path)]

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device)   # (B, H, W)  long
        logits = seg_model(images)           # (B, 3, H, W)
        preds  = logits.argmax(dim=1)        # (B, H, W)

        for i in range(len(images)):
            pred_i = preds[i].cpu().numpy()  # (H, W)
            mask_i = masks[i].cpu().numpy()  # (H, W)
            ious = []
            for cls_id in [1, 2]:  # normal, canker (skip bg)
                inter = ((pred_i == cls_id) & (mask_i == cls_id)).sum()
                union = ((pred_i == cls_id) | (mask_i == cls_id)).sum()
                if union > 0:
                    ious.append(inter / union)
            mean_iou = float(np.mean(ious)) if ious else 0.0
            results_per_image.append((mean_iou, batch["image_path"][i]))

# 정렬 → 하위 3개
results_per_image.sort(key=lambda x: x[0])
print("IoU 하위 3개:")
for iou, p in results_per_image[:3]:
    print(f"  {Path(p).name:40s}  mean_IoU(non-bg)={iou:.4f}")

# IoU 분포 히스토그램
ious_all = [r[0] for r in results_per_image]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ious_all, bins=20, edgecolor="white", color="steelblue")
ax.set_title("Per-image Mean IoU (non-bg) — val set")
ax.set_xlabel("Mean IoU")
ax.set_ylabel("이미지 수")
plt.tight_layout()
plt.show()


In [ ]:
# worst-3 시각화: original | GT mask | pred mask
PALETTE = np.array([
    [0,   0,   0  ],   # 0: bg      → black
    [100, 180, 100],   # 1: normal  → green
    [220, 60,  60 ],   # 2: canker  → red
], dtype=np.uint8)

def _mask_to_rgb(mask_2d):
    """(H, W) int → (H, W, 3) uint8 with class palette."""    rgb = PALETTE[mask_2d.clip(0, 2)]
    return rgb

worst3 = results_per_image[:3]
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

with torch.no_grad():
    for row_idx, (iou_val, img_path) in enumerate(worst3):
        # dataset item 재로드 (transform 적용된 텐서)
        for ip, jp in val_ds.items:
            if str(ip) == img_path:
                item = val_ds[val_ds.items.index((ip, jp))]
                break
        img_tensor = item["image"].unsqueeze(0).to(device)  # (1, 3, H, W)
        gt_mask    = item["mask"].numpy()                    # (H, W)

        logits = seg_model(img_tensor)
        pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy()  # (H, W)

        # 역정규화 (ImageNet stats)
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img_np = item["image"].permute(1, 2, 0).numpy()  # (H, W, 3) float
        img_np = (img_np * std + mean).clip(0, 1)

        axes[row_idx, 0].imshow(img_np)
        axes[row_idx, 0].set_title(f"원본\n{Path(img_path).name}", fontsize=8)
        axes[row_idx, 0].axis("off")

        axes[row_idx, 1].imshow(_mask_to_rgb(gt_mask))
        axes[row_idx, 1].set_title(f"GT Mask", fontsize=8)
        axes[row_idx, 1].axis("off")

        axes[row_idx, 2].imshow(_mask_to_rgb(pred_mask))
        axes[row_idx, 2].set_title(f"Pred Mask\nIoU={iou_val:.3f}", fontsize=8)
        axes[row_idx, 2].axis("off")

plt.suptitle("IoU 하위 3개 (실패 케이스)", fontsize=14)
plt.tight_layout()
plt.show()


## 6. Class별 성능 비교

bg를 제외한 normal / canker 두 클래스의 IoU와 Dice를 바 차트로 비교합니다.


In [ ]:
# bg 제외, normal(idx=1), canker(idx=2)
iou_normal  = metrics["iou_per_class"][1]
iou_canker  = metrics["iou_per_class"][2]
dice_normal = metrics["dice_per_class"][1]
dice_canker = metrics["dice_per_class"][2]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# --- IoU 바 차트 ---
ax = axes[0]
bars = ax.bar(["normal", "canker"], [iou_normal, iou_canker],
              color=["steelblue", "tomato"], width=0.4)
for bar, val in zip(bars, [iou_normal, iou_canker]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", va="bottom", fontsize=11)
ax.set_ylim(0.80, 1.01)
ax.set_title("Class별 IoU (bg 제외)")
ax.set_ylabel("IoU")
ax.grid(axis="y", alpha=0.3)

# --- Dice 바 차트 ---
ax2 = axes[1]
bars2 = ax2.bar(["normal", "canker"], [dice_normal, dice_canker],
                color=["steelblue", "tomato"], width=0.4)
for bar, val in zip(bars2, [dice_normal, dice_canker]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
             f"{val:.4f}", ha="center", va="bottom", fontsize=11)
ax2.set_ylim(0.90, 1.005)
ax2.set_title("Class별 Dice (bg 제외)")
ax2.set_ylabel("Dice")
ax2.grid(axis="y", alpha=0.3)

plt.suptitle(f"Segmentation 성능: normal vs canker", fontsize=13)
plt.tight_layout()
plt.show()

print(f"IoU  — normal={iou_normal:.4f}, canker={iou_canker:.4f}, gap={iou_normal - iou_canker:.4f}")
print(f"Dice — normal={dice_normal:.4f}, canker={dice_canker:.4f}, gap={dice_normal - dice_canker:.4f}")
print()
print("canker IoU가 낮은 이유:")
print("  - val set의 canker 이미지가 29장으로 normal(59장)의 절반 수준 → 희귀 클래스 학습 어려움")
print("  - confusion_matrix에서 canker→normal 오분류 비율이 높음 (두 클래스의 색상/질감 유사)")


## 7. 관찰 요약

- **빠른 수렴**: epoch 1에서 mIoU = 0.54(canker class IoU = 0.000)로 시작했지만, epoch 2에서 0.91, epoch 3에서 0.94로 급격히 개선됩니다.
  이는 ResNet34 ImageNet pretrain의 강력한 초기화 덕분으로, fine-tuning이 매우 효율적으로 이루어졌음을 시사합니다.
- **canker IoU 격차**: normal IoU = 0.937, canker IoU = 0.886으로 약 0.05 차이가 납니다.
  val set의 canker 이미지 수(29장)가 normal(59장)의 절반에 불과해 희귀 클래스 학습이 어렵습니다.
- **Polygon = 과일 외곽선**: 라벨의 polygon이 병변 자체가 아닌 과일 전체 외곽선이므로, 실제로는 과일을 클래스(정상 vs 궤양병)로 구분하는 문제입니다.
  이 덕분에 mask 경계가 명확하여 3 epoch의 짧은 학습에도 높은 mIoU가 가능했습니다.
- **bg IoU = 0.998**: 배경(흰색 스튜디오)이 매우 균일해 bg 분할은 trivial합니다. 전체 mIoU를 올려주는 효과가 있습니다.


## 📝 Your turn

아래 질문에 답하는 분석을 직접 추가해보세요.

1. **궤양병(canker) IoU 0.886 vs 정상 IoU 0.937 — 격차의 원인?**
   confusion matrix에서 canker→normal 오분류 픽셀 수와 normal→canker 오분류 픽셀 수를 계산하고,
   어느 방향이 더 많은지 정량화해 격차의 주 원인을 분석해보세요.

2. **데이터에서 polygon이 전체 과일 외곽선이지 병변이 아니다. 이것이 'segmentation' 난이도를 낮춘 요인일까?**
   실제 병변(canker spot) 단위로 다시 annotation한다면 mIoU는 얼마나 떨어질까요?
   관련 논문을 찾아 병변 수준 segmentation의 전형적인 IoU 수치와 비교해보세요.

3. **encoder를 efficientnet-b0로 바꾸면 IoU가 얼마나 떨어질까? FPS는?**
   `build_model(encoder_name="efficientnet-b0")`로 재학습하고 IoU / latency trade-off를 비교해보세요.
   파라미터 수도 함께 출력하세요 (`sum(p.numel() for p in model.parameters())`).

4. **DeepLabV3+ 등 더 무거운 decoder를 써야 할 만큼 어려운 문제인가?**
   U-Net이 이미 0.94 mIoU인 상황에서 `smp.DeepLabV3Plus` 등을 시도할 실익이 있을까요?
   SOTA 모델 대비 성능 상한(ceiling)을 이 데이터셋에서 어떻게 추정할 수 있을지 논의해보세요.

5. **실제 임상/현장 활용을 위해 픽셀 단위 분할이 꼭 필요한가? 분류 + bbox로 충분?**
   농가 현장에서 스마트폰 앱으로 감귤 궤양병을 진단하는 시나리오를 가정하고,
   분류(P1), 탐지(P2), 분할(P3) 각각의 UX 활용성과 compute cost를 비교하여 최적 파이프라인을 제안해보세요.
